In [1]:
from pathlib import Path
from neuralnetwork import NeuralNetwork
from semanticreasoning import ProcessData
import torch
import os
import time
import csv
import numpy as np

c:\ProgramData\anaconda3\envs\robotics\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from hierarchical_network import HierarchicalEmbodimentAgnosticNetwork

In [3]:
# nn_model = NeuralNetwork.loadModel(384, 3, "sentence_nn.pth")
dataProcessor = ProcessData()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4305.49it/s]


In [4]:
# # if Path("sentence_command.txt").is_file():
# last_modified_time = os.path.getmtime("sentence_command.txt")
# while True:
#     current_modified_time = os.path.getmtime("sentence_command.txt")
#     if current_modified_time != last_modified_time:
#         with open("sentence_command.txt", "r") as f:
#             sentence = f.readline().strip()

#         embedding = dataProcessor.getEmbeddings(sentence)

#         embedding = torch.tensor(
#             embedding,
#             dtype=torch.float32
#         ).unsqueeze(0)   # make shape (1,input_size)

#         nn_model.model.eval()

#         with torch.no_grad():
#             commands = nn_model.model(embedding)
#         commands = commands.squeeze().tolist()

#         BASE = Path.cwd().parent
#         cmd = BASE / "experiments" / "commands.txt"
#         if cmd.is_file():
#             with open(cmd, "w") as f:
#                 for value in commands:
#                     if value > 1:
#                         value = 1.0
#                     elif value < -1:
#                         value = -1.0
#                     f.write(f"{value}\n")
#             continue
#         else:
#             break
#     else:
#         continue

In [5]:
BASE = Path.cwd().parent
desc_vector_dict = {}
robot_names = ["AnymalB", "AnymalC", "Badger", "UnitreeA1", "UnitreeGo1", "UnitreeGo2", "UnitreeH1", "RobotisOP3", "Cassie"]

for robot_name in robot_names:
    file_path = BASE / "experiments" / f"description_vectors/{robot_name}_description_vectors.csv"

    if file_path.is_file():
        #current_robot = robot_name
        with open(file_path, "r") as file:
            
            reader = csv.reader(file)

            next(reader)

            list_vectors = []
            for row in reader:
                vector = []
                for vect in row[1:]:
                    vector.append((float(vect)))
                list_vectors.append(vector)
                    

            desc_vector_dict[robot_name] = torch.tensor(np.array(list_vectors, dtype=np.float32))


In [6]:
with open(f"description_vectors/UnitreeGo1_description_vectors.csv", "r") as file:
        
    reader = csv.reader(file)

    next(reader)

    list_vectors = []
    for row in reader:
        vector = []
        for vect in row[1:]:
            vector.append((float(vect)))
        list_vectors.append(vector)
            
    desc_vector_dict = {}
    desc_vector_dict["UnitreeGo1"] = torch.tensor(np.array(list_vectors, dtype=np.float32))

In [7]:
network = HierarchicalEmbodimentAgnosticNetwork()
network.load_networks()

In [9]:
current_robot = "UnitreeGo1"
last_modified_time = os.path.getmtime("sentence_command.txt")
robot_desc_vector = desc_vector_dict[current_robot]


while True:
    current_modified_time = os.path.getmtime("sentence_command.txt")
    if current_modified_time != last_modified_time:
        last_modified_time = current_modified_time
        with open("sentence_command.txt", "r") as f:
            sentence = f.readline().strip()
        
        embedding = dataProcessor.getEmbeddings(sentence)

        embedding = torch.tensor(
            embedding,
            dtype=torch.float32
        ).unsqueeze(0)   # make shape (1,input_size)

        commands = network.get_commands(robot_desc_vector, embedding)
        commands = commands.squeeze().detach().cpu().numpy()

        
        cmd = BASE / "experiments" / "commands.txt"
        if cmd.is_file():
            for i in [0,3,6,9,12,15]:
                with open(cmd, "w") as file:
                    file.write(f"{commands[i]}\n{commands[i+1]}\n{commands[i+2]}\n")
                time.sleep(float(commands[18])+1)
        else:
            break
    time.sleep(0.05) 




KeyboardInterrupt: 

In [ ]:
Path.cwd().parent

: 

: 